
# Lab 9: Build a Log Aggregator

In this lab, you will create your own log generator, build a command-line utility that scans log files, summarizes their contents, and provides insight into system behavior. Data structures to track log message levels such as `INFO`, `WARNING`, `ERROR`, and `CRITICAL`.

This lab reinforces:
- File I/O
- Pattern recognition (regex)
- Dictionaries and counters
- Functions and modularity
- Optional: CLI arguments, logging



## Part 1: Create Log files (20%)
Using the the following example log format below create a **python file** that will log errors In a structured tree format 

You will find examples in the folder called Logs that you can use to build your program.

Remember set of logs should have a varied levels of log entries (`INFO`, `WARNING`, `ERROR`, `CRITICAL`) and tailored message types for different service components.
You must create 5 structured logs here are some examples:

    sqldb
    ui
    frontend.js
    backend.js
    frontend.flask
    backend.flask

You may use chat GPT to create sample outputs NOT THE LOGS. IE:

    System failure
    Database corruption
    Disk failure detected
    Database corruption


#### Joanna's work for Part 1

I decided to use a json file to loop through a few different options for the errors. The way the log is configured will only return one error for each day for each description in this format: `application.info(error_log_json("info"))`. The json file had to be in a specific folder in order for the code to work, so below is the representation of the json file.

In [1]:
#json file contents
{"critical":{
    "1": "Imminent Shut Down",
    "2": "Memory Overload",
    "3": "No disk space",
    "4": "Computer on fire"
    }, 
"warning": {
    "1": "Computer overheating",
    "2": "Low disk space", 
    "3": "Computer fell over",
    "4": "Low battery"
    },
"error":{
    "1": "No connection",
    "2": "File not found", 
    "3": "link to other page is dead",
    "4": "Low battery"
    },
"info":{
    "1": "User logged in",
    "2": "Application starting", 
    "3": "User logged out",
    "4": "New file created"
    }    
}

{'critical': {'1': 'Imminent Shut Down',
  '2': 'Memory Overload',
  '3': 'No disk space',
  '4': 'Computer on fire'},
 'warning': {'1': 'Computer overheating',
  '2': 'Low disk space',
  '3': 'Computer fell over',
  '4': 'Low battery'},
 'error': {'1': 'No connection',
  '2': 'File not found',
  '3': 'link to other page is dead',
  '4': 'Low battery'},
 'info': {'1': 'User logged in',
  '2': 'Application starting',
  '3': 'User logged out',
  '4': 'New file created'}}

In [ ]:
## for importing and running the logging file
# import log_part_1
# log_part_1.main()

# code in file log_part_1.py
import logging
import logging.handlers
from datetime import timedelta, datetime
import time
import freezegun
import json
import random

# configurations for application
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')
application = logging.getLogger("logger_application")
application.setLevel(logging.INFO)
application.setLevel(logging.WARNING)
application.setLevel(logging.ERROR)
application.setLevel(logging.CRITICAL)

a_handler = logging.handlers.TimedRotatingFileHandler(
        filename = "application_archived_log.log",
        when = "D",
        backupCount = 3,
    )

# configuration for ui
ui = logging.getLogger("logger_application.ui")
ui.setLevel(logging.CRITICAL)
ui.setLevel(logging.INFO)
ui.setLevel(logging.WARNING)
ui.setLevel(logging.ERROR)

ui_handler = logging.handlers.TimedRotatingFileHandler(
        filename = "ui_archived_log.log",
        when = "D",
        backupCount = 3,
    )

# configuration for utils
utils = logging.getLogger("logger_application.ui.utils")
utils.setLevel(logging.CRITICAL)
utils.setLevel(logging.INFO)
utils.setLevel(logging.WARNING)
utils.setLevel(logging.ERROR)

utils_handler = logging.handlers.TimedRotatingFileHandler(
        filename = "utils_archived_log.log",
        when = "D",
        backupCount = 3,
    )

# configuration for frontend
frontend = logging.getLogger("logger_application.ui.utils.frontend")
frontend.setLevel(logging.CRITICAL)
frontend.setLevel(logging.WARNING)
frontend.setLevel(logging.ERROR)
frontend.setLevel(logging.INFO)

frontend_handler = logging.handlers.TimedRotatingFileHandler(
        filename = "frontend_archived_log.log",
        when = "D",
        backupCount = 3,
    )

# configuration for backend
backend = logging.getLogger("logger_application.ui.utils.backend")
backend.setLevel(logging.CRITICAL)
backend.setLevel(logging.WARNING)
backend.setLevel(logging.ERROR)
backend.setLevel(logging.INFO)

backend_handler = logging.handlers.TimedRotatingFileHandler(
        filename = "backend_archived_log.log",
        when = "D",
        backupCount = 3,
    )

# initializing formatter
formatting = logging.Formatter(
    fmt = ("%(asctime)s | %(name)s | %(levelname)s | %(message)s"
    )
)

#adding handlers
a_handler.setFormatter(formatting)
ui_handler.setFormatter(formatting)
utils_handler.setFormatter(formatting)
frontend_handler.setFormatter(formatting)
backend_handler.setFormatter(formatting)
application.addHandler(a_handler)
ui.addHandler(ui_handler)
utils.addHandler(utils_handler)
frontend.addHandler(frontend_handler)
backend.addHandler(backend_handler)

#trying to use JSON to read/generate error messages
def error_log_json(level):
    with open("Labs\Lab 9\error_dictionary.json") as f:
        file = json.load(f)

    x = str(random.randint(1,4)) # the num of subkeys in the dict. If you add more options, just update.

    for key, value in file[level].items():
        if key == x:
            return file[level][x]

# freezegun function
def main():
    with freezegun.freeze_time() as frozen:
        for i in range(10):
            frozen.tick(timedelta(hours = 24))
            time.sleep(0.1)

            # These guys print the level and error name
            application.info(error_log_json("info"))
            ui.critical(error_log_json("critical"))
            utils.error(error_log_json("error"))
            frontend.warning(error_log_json("warning"))
            backend.critical(error_log_json("critical"))



### Example Log Format

You will work with logs that follow this simplified structure:

```
2025-04-11 23:20:36,913 | my_app | INFO | Request completed
2025-04-11 23:20:36,914 | my_app.utils | ERROR | Unhandled exception
2025-04-11 23:20:36,914 | my_app.utils.db | CRITICAL | Disk failure detected
```


#### Joanna's work

I was able to successfully get the logger into the following format. The heierarchical errors did work, and I was able to get the names of the files into the logging format too.

As of 4/20/2025 at 11:14 PM, here is a representation of my application_archived_log.log file:

```
2025-04-27 23:07:59,964 | logger_application.ui | CRITICAL | Computer on fire
2025-04-27 23:07:59,964 | logger_application.ui.utils | ERROR | No connection
2025-04-27 23:07:59,964 | logger_application.ui.utils.frontend | WARNING | Low disk space
2025-04-27 23:07:59,964 | logger_application.ui.utils.backend | CRITICAL | Memory Overload
2025-04-28 23:07:59,964 | logger_application.ui | CRITICAL | Imminent Shut Down
2025-04-28 23:07:59,964 | logger_application.ui.utils | ERROR | File not found
2025-04-28 23:07:59,964 | logger_application.ui.utils.frontend | WARNING | Low disk space
2025-04-28 23:07:59,964 | logger_application.ui.utils.backend | CRITICAL | Computer on fire
2025-04-29 23:07:59,964 | logger_application.ui | CRITICAL | No disk space
2025-04-29 23:07:59,964 | logger_application.ui.utils | ERROR | link to other page is dead
2025-04-29 23:07:59,964 | logger_application.ui.utils.frontend | WARNING | Computer fell over
2025-04-29 23:07:59,964 | logger_application.ui.utils.backend | CRITICAL | No disk space
2025-04-30 23:07:59,964 | logger_application.ui | CRITICAL | No disk space
2025-04-30 23:07:59,964 | logger_application.ui.utils | ERROR | File not found
2025-04-30 23:07:59,964 | logger_application.ui.utils.frontend | WARNING | Computer overheating
2025-04-30 23:07:59,964 | logger_application.ui.utils.backend | CRITICAL | Computer on fire

```

## Part 2: Logging the Log File (40%)
    New File
### Part 2a: Read the Log File (see lab 7) (10%)


Write a function to read the contents of a log file into a list of lines. Handle file errors gracefully.

### Part 2b: Parse Log Lines (see code below if you get stuck) (10%)

Use a regular expression to extract:
- Timestamp
- Log name
- Log level
- Message

### Part 2c: Count Log Levels (20%)

Create a function to count how many times each log level appears. Store the results in a dictionary. Then output it as a Json File
You may pick your own format but here is an example. 
```python
{
    "INFO": 
    {
        "Request completed": 42, 
        "Heartbeat OK": 7
    }

    "WARNING":
    {
        ...
    }
}

```


In [ ]:
# Paste your python file here don't for get to upload it with your submission
import re

def read_log_line(log_filepath):
    # structure taken from Mr. Power's Lecture 6 notes
    try: 
        with open(log_filepath, "r") as file:
            lines = file.readlines()
        return lines
    except:
        FileNotFoundError

def parse_log_line(line_list):
    match_list = []
    
    for line in line_list:
        # print(f"{line}") # it can go thru all lines. line var considered a string

        pattern = r"^(.*?)\s\|\s(\w+)\s\|\s(\w+)\s\|\s(.*)$"
        match = re.match(pattern, line) #this tries to match each entire line
        
        if match: #this part modified from ChatGPT
           match_list.append(match.groups()) 

def count_log_levels(parsed):
    count_dict = {"INFO": {}, "WARNING":{}, "ERROR":{}, "CRITICAL":{}}

    for tuple in parsed:
        level = tuple[2]
        message = tuple[3]
        count_i, count_e, count_w, count_c = 0

        if level == "INFO":
            count_dict["INFO"][message] = count_i + 1
        elif level == "ERROR":
            count_dict["ERROR"][message] = count_e + 1
        elif level == "WARNING":
            count_dict["WARNING"][message] = count_w + 1
        elif level == "CRITICAL":
            count_dict["CRITICAL"][message] = count_c + 1

    filename = 'log_level_count.json'
    with open(filename,'w') as f:
        f.write(count_dict)



## Step 3: Generate Summary Report (40%)
    New File
### Step 3a (20%):
 Develop a function that continuously monitors your JSON file(s) and will print a real-time summary of log activity. It should keep count of the messages grouped by log level (INFO, WARNING, ERROR, CRITICAL) and display only the critical messages. (I.e. If new data comes in the summary will change and a new critical message will be printed)
 - note: do not reprocess the entire file on each update.  

### Step 3a: Use a Matplotlib (Lecture 10) (20%)
Develop a function that continuously monitors your JSON file(s) and will graph in real-time a bar or pie plot of each of the errors.  (a graph for each log level). 
- The graph should show the distribution of log messages by level  (INFO, WARNING, ERROR, CRITICAL)  


### Critical notes:
- Your code mus use Daemon Threads (Lecture 14)
- 3a and 3b do not need to run at the same time. 


In [ ]:
# Paste your python file here 
import matplotlib

def jsonMonitor(file):
    with open(file, "r") as f:
        count_dict = f.load()

    count_i, count_e, count_w, count_c = 0
    # critical_list = []
    for key in count_dict:
        if key == "INFO":
            count_i += 1
        elif key == "WARNING":
            count_w += 1
        elif key == "ERROR":
            count_e += 1
        else:
            count_c += 1
            for value in key:
                print(count_dict[key][value])

def graphErrors(file):
    with open(file, "r") as f:
        count_dict = f.load()

    #taken from Mr. Powers's notes
    categories = list("INFO","ERROR","WARNING","CRITICAL")
    


    matplotlib.bar(categories, values)
    matplotlib.title('Bar Plot')
    matplotlib.xlabel('Categories')
    matplotlib.ylabel('Values')

    matplotlib.show()


In [ ]:
# Here is a sample regex that parses a log file and extracts relevant information. 
# you will need to modify it. Review Lecture 11
import re

def parse_log_line(line):
    pattern = r"^(.*?)\s\|\s(\w+)\s\|\s(\w+)\s\|\s(.*)$"
    match = re.match(pattern, line)
   
